# DeFiPy: Python SDK for DeFi Analytics
## Chapter 9: Building Autonomous DeFi Agents

### Listing 9.1: Base Agent Interface

In [1]:
class BaseAgent:
    def check_condition(self) -> bool:
        """Determine if the execution criteria are met."""
        raise NotImplementedError("Condition logic must be implemented.")

    def execute_action(self):
        """Execute the strategy when the condition is true."""
        raise NotImplementedError("Execution logic must be implemented.")

    def run(self):
        """Control loop or trigger mechanism."""
        if self.check_condition():
            self.execute_action()

### Listing 9.2: Price Threshold Agent

In [2]:
from defipy import UniswapFactory, UniswapExchangeData, Swap, Join, ERC20

class PriceThresholdSwapAgent(BaseAgent):
    def __init__(self, token0: ERC20, token1: ERC20, fee: int, threshold: float):
        factory = UniswapFactory("Pool factory", "0x2")
        exch_data = UniswapExchangeData(tkn0=token0, tkn1=token1, symbol="LP", address="0x3")
        self.lp = factory.deploy(exch_data)
        self.threshold = threshold

    def init(self, tkn0_amt, tkn1_amt):
        join = Join()
        join.apply(self.lp, "user", tkn0_amt, tkn1_amt)

    def check_condition(self, tkn) -> bool:
        price = self.lp.get_price(tkn)
        return price >= self.threshold

    def execute_action(self, tkn, user_nm, tkn_amt):
        swap = Swap()
        tkn_amt_out = swap.apply(self.lp, tkn, user_nm, tkn_amt)
        tkn_nm_out = self.lp.token1 if tkn.token_name == self.lp.token0 else self.lp.token0
        print(f"Threshold met. Executing swap {tkn_amt} {tkn.token_name} for {tkn_amt_out:.4f} {tkn_nm_out}")
        return  

    def run(self, tkn, user_nm, tkn_amt):
        """Control loop or trigger mechanism."""
        if self.check_condition(tkn):
            self.execute_action(tkn, user_nm, tkn_amt)
        else:
            print('Threshold not met')

### Listing 9.3: Pydantic Configuration

In [3]:
from pydantic import BaseModel, ConfigDict
from defipy import ERC20

# code block from 9.2

class PriceThresholdConfig(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True) 
    token0: ERC20
    token1: ERC20
    fee: int
    threshold: float

eth = ERC20("ETH", "0xAAA...")
usdc = ERC20("USDC", "0xBBB...")

config = PriceThresholdConfig(token0 = eth, token1 = usdc, fee=3000, threshold=3200.0)

### Listing 9.5: Instantiating Agent with Config

In [4]:
# code block from 9.3

agent = PriceThresholdSwapAgent(**config.dict())
agent.init(100, 400000)
agent.run(eth, "user", 1)

Threshold met. Executing swap 1 ETH for 3948.6321 USDC


### Listing: 9.6 Price Threshold Configuration

In [5]:
from pydantic import BaseModel

class PriceThresholdConfig(BaseModel):  
    threshold: float  # e.g., 3000.0 (price above which to swap)
    swap_amount: float  # e.g., 1.0 (swap amount if threshold met)
    pool_address: str  # Uniswap V2 pool
    provider_url: str  # e.g., Infura for Web3Scout
    abi_name: str  # e.g., 'UniswapV2Pair' (new field for ABI identifier)
    platform: str  # e.g., 'UNI' or 'SUSHI' for the protocoll

### Listing 9.7: Instantiate agent

In [9]:
from defipy import *
from web3scout import *

price_threshold = 3000.0
swap_amount = 1.0
pair_address = "0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"
provider_url = "https://mainnet.infura.io/v3/9624e3e5c40f4ac3958b79fa5aa2562d"
platform = Platform.AGNOSTIC
abi_name = JSONContract.UniswapV2Pair

config = PriceThresholdConfig(
    threshold = price_threshold,
    swap_amount = swap_amount,
    pool_address = pair_address,
    provider_url = provider_url,
    platform = platform,
    abi_name = abi_name,
)

agent = PriceThresholdSwapAgent(config)

print(f"Monitoring price movements @ pool address {pair_address}")

Monitoring price movements @ pool address 0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc


### Listing 9.8: Key Methods of PriceThresholdSwapAgent

In [10]:
def get_token_price(self, tkn1_over_tkn0 = True, block_num = None):
        
    if(block_num == None):
        reserves = self.lp_data.reserves
    else:
        lp_contract = self._init_lp_contract()
        reserves = lp_contract.functions.getReserves().call(block_identifier=block_num)
            
    price = self.calc_price(reserves[0], reserves[1], tkn1_over_tkn0)
    
    return price

def check_condition(self, threshold = None, tkn1_over_tkn0 = True, block_num = None):
    self.config.threshold = self.config.threshold if threshold == None else threshold;
    self.apply()
    price = self.get_token_price(tkn1_over_tkn0, block_num)
    return price > self.config.threshold

def apply(self):
    self.lp_contract = self._init_lp_contract()
    
    reserves = self.lp_contract.functions.getReserves().call()
    token0_address = self.lp_contract.functions.token0().call()
    token1_address = self.lp_contract.functions.token1().call()
    reserve0 = reserves[0]; reserve1 = reserves[1]

    w3 = self.connector.get_w3()
    FetchERC20 = FetchToken(w3)
    TKN0 = FetchERC20.apply(token0_address)
    TKN1 = FetchERC20.apply(token1_address)

    self.lp_data = UniswapPoolData(TKN0, TKN1, reserves)

def run_batch(self, tkn, events):
    start_block = events[0]['blockNumber']
    lp = self.prime_pool_state(start_block, 'user')

    """Fetch batch of Sync events and process sequentially."""
    if not events:
        print("No Sync events found in range.")
        return
    for k in events:
        reserve0 = events[k]['args']['reserve0']
        reserve1 = events[k]['args']['reserve1']
        block_num = events[k]['blockNumber']
        event_price = self.calc_price(reserve0, reserve1, tkn1_over_tkn0 = True)
        self.execute_action(lp, tkn, event_price, block_num)

### Listing 9.9: Running the Price Threshold Swap Agent

In [11]:
# Apply agent
agent.apply()

# Grab agent data
tkn1_over_tkn0 = True
price = agent.get_token_price(True)
price_condition_pass = agent.check_condition()
contract_instance = agent.get_contract_instance()
lp_data = agent.get_lp_data()

tkn0 = lp_data.tkn0; tkn1 = lp_data.tkn1; reserves = lp_data.reserves

# Print agent data
print("---------------------------------------------------------------------------------------")
print(f"Agent data @ pool address {pair_address}")
print("---------------------------------------------------------------------------------------")
print(f"reserve0 = {reserves[0]/(10**tkn0.token_decimal):.2f} {tkn0.token_name} @ token address {tkn0.token_addr}")
print(f"reserve1 = {reserves[1]/(10**tkn1.token_decimal):.2f} {tkn1.token_name} @ token address {tkn1.token_addr}")

if(tkn1_over_tkn0):
    price = (reserves[0] / reserves[1]) * (10 ** (tkn1.token_decimal - tkn0.token_decimal))
    print(f"\n{tkn1.token_name} Price in {tkn0.token_name}: {price}")
    print(f"Threshold PASS, {tkn1.token_name} Price > {price_threshold}: {price_condition_pass}")
else:
    price = (reserves[1] / reserves[0]) * (10 ** (tkn0.token_decimal - tkn1.token_decimal))
    print(f"\n{tkn0.token_name} Price in {tkn1.token_name}: {price}")
    print(f"Threshold PASS, {tkn0.token_name} Price > {price_threshold}: {price_condition_pass}")

---------------------------------------------------------------------------------------
Agent data @ pool address 0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc
---------------------------------------------------------------------------------------
reserve0 = 11311997.11 USDC @ token address 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
reserve1 = 3501.02 WETH @ token address 0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2

WETH Price in USDC: 3231.0613952681706
Threshold PASS, WETH Price > 3000.0: True


### Listing 9.13: Volume Spike Configuration

In [12]:
from pydantic import BaseModel

class VolumeSpikeConfig(BaseModel):
    volume_threshold: float  # Volume threshold for notification (e.g., USD value of trades)
    pool_address: str       # Uniswap V2 pool address
    provider_url: str       # Web3 provider URL (e.g., Infura)
    abi_name: str  # e.g., 'UniswapV2Pair' (new field for ABI identifier)
    platform: str  # e.g., 'UNI' or 'SUSHI' for the protocoll
    user_position: float  # User's LP shares or amount

### Listing 9.14: Instantiate agent

In [13]:
from defipy import *
from web3scout import *

volume_threshold = 1000
pair_address = "0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"
provider_url = "https://mainnet.infura.io/v3/9624e3e5c40f4ac3958b79fa5aa2562d"
platform = Platform.AGNOSTIC
abi_name = JSONContract.UniswapV2Pair
user_position = 10

config = VolumeSpikeConfig(
    volume_threshold = volume_threshold,
    pool_address = pair_address,
    provider_url = provider_url,
    platform = platform,
    abi_name = abi_name,
    user_position = user_position
)

agent = VolumeSpikeNotifierAgent(config)
agent.init()

print(f"Monitoring TVL changes @ pool address {pair_address}")

Monitoring TVL changes @ pool address 0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc


### Listing 9.15: Key Methods of VolumeSpikeNotifierAgent

In [14]:
def get_pool_volume(self, lp, tkn, block_num):
    """Calculate TVL from reserves (sum in USD, assuming base_token normalization)."""

    tkn0 = self.get_lp_data().tkn0
    tkn1 = self.get_lp_data().tkn1
    prev_tkn0 = lp.get_reserve(tkn0)
    prev_tkn1 = lp.get_reserve(tkn1)

    lp = self.update_mock_pool(lp, block_num)
    
    dtkn0 = abs(lp.get_reserve(tkn0) - prev_tkn0)
    dtkn1 = abs(lp.get_reserve(tkn1) - prev_tkn1)

    if(tkn.token_name == tkn0.token_name):
        volume = dtkn0 + LPQuote().get_amount(lp, tkn1, dtkn1)  
    elif(tkn.token_name == tkn1.token_name):
        volume = dtkn1 + LPQuote().get_amount(lp, tkn0, dtkn0)
    
    self.pool_volume = volume
    return volume

def check_condition(self, lp, tkn, threshold, block_num = None):
    """Check if TVL is below threshold."""
    block_num = self.get_w3().eth.block_number if block_num == None else block_num
    volume = self.get_pool_volume(lp, tkn, block_num)
    return volume > threshold

def apply(self, lp, tkn, user_nm, block_num):
    """Execute liquidity exit if condition met."""
    if self.check_condition(lp, tkn, self.config.volume_threshold, block_num):
        vol = self.pool_volume
        print(f"Block {block_num}: Volume ({tkn.token_name}) = {vol}, outside threshold {self.config.volume_threshold}")
        return vol
    else:
        print(f"Block {block_num}: Volume threshold condition met for {lp.name} LP")
        return None

### Listing 9.16: Running the Volume Spike Notifier Agent

In [15]:
abi = ABILoad(platform, abi_name)
connect = ConnectW3(provider_url)
connect.apply()

last_block = connect.get_w3().eth.block_number
start_block = last_block - 100

# Grab batch sync events from pool
rEvents = RetrieveEvents(connect, abi)
events = rEvents.apply(EventType.SWAP, address = pair_address, start_block=start_block, end_block=last_block)
df_events = rEvents.to_dataframe(events)
df_events.head(2)

tkn0 = agent.get_lp_data().tkn0
tkn1 = agent.get_lp_data().tkn1
lp = agent.prime_mock_pool(start_block, 'user')
lp.summary()

agent.run_batch(lp, tkn0, 'user', events)

Exchange USDC-WETH (LP)
Reserves: USDC = 11326668.8652, WETH = 3496.466554982209
Liquidity: 0.07280779164560718 

Block 23729614: Volume threshold condition met for USDC-WETH LP
Block 23729614: Volume threshold condition met for USDC-WETH LP
Block 23729614: Volume threshold condition met for USDC-WETH LP
Block 23729623: Volume threshold condition met for USDC-WETH LP
Block 23729630: Volume (USDC) = 1274.585600564262, outside threshold 1000.0
Block 23729662: Volume threshold condition met for USDC-WETH LP
Block 23729670: Volume threshold condition met for USDC-WETH LP
Block 23729672: Volume threshold condition met for USDC-WETH LP
Block 23729699: Volume (USDC) = 16691.54691050841, outside threshold 1000.0
Block 23729709: Volume (USDC) = 12130.39328997525, outside threshold 1000.0
Block 23729710: Volume (USDC) = 14649.085465621836, outside threshold 1000.0


### Listing 9.17: Impermanent Loss Configuration

In [16]:
from pydantic import BaseModel

class ImpermanentLossConfig(BaseModel):
    il_threshold: float  # Impermanent loss threshold percentage (e.g., 5.0 for 5%)
    pool_address: str       # Uniswap V2 pool address
    provider_url: str       # Web3 provider URL (e.g., Infura)
    abi_name: str  # e.g., 'UniswapV2Pair' (new field for ABI identifier)
    platform: str  # e.g., 'UNI' or 'SUSHI' for the protocoll
    user_position: float    # Initial mock position amount for off-chain testing
    exit_percentage: float  # Percentage of position to exit upon trigger (e.g., 1.0 for 100%)

### Listing 9.18: Instantiate agent

In [17]:
from defipy import *
from web3scout import *

il_threshold = 99.70
pair_address = "0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"
provider_url = "https://mainnet.infura.io/v3/9624e3e5c40f4ac3958b79fa5aa2562d"
platform = Platform.AGNOSTIC
abi_name = JSONContract.UniswapV2Pair
user_position = 100
exit_percentage = 5

config = ImpermanentLossConfig(
    il_threshold = il_threshold,
    pool_address = pair_address,
    provider_url = provider_url,
    platform = platform,
    abi_name = abi_name,
    user_position = user_position,
    exit_percentage = exit_percentage
)

agent = ImpermanentLossAgent(config)
agent.init()

print(f"Monitoring TVL changes @ pool address {pair_address}")

Monitoring TVL changes @ pool address 0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc


### Listing 9.19: Key Methods of ImpermanentLossExitAgent

In [18]:
def get_impermanent_loss(self) -> float:
    """Calculate impermanent loss percentage based on initial and current reserves."""
    returns_calc = self.iLoss.apply(fees = True)
    return returns_calc

def check_condition(self, tkn, threshold):
    """Check if TVL is below threshold."""
    position_value = self.get_current_position_value(tkn)
    return position_value < threshold

def apply(self, lp, tkn, user_nm, block_num):
    """Execute liquidity exit if condition met."""
    self.update_mock_pool(lp, block_num)
    if self.check_condition(tkn, self.config.il_threshold):
        val = self.get_current_position_value(tkn)
        print(f"Block {block_num}: Value ({tkn.token_name}) = {val}, outside loss threshold {self.config.il_threshold}")
        return val
    else:
        print(f"Block {block_num}: Value threshold condition met for {lp.name} LP")
        return None

def run_batch(self, lp, tkn, user_nm, events: dict):
    """Process batched Sync events to check TVL and trigger exits."""
    if not events:
        print("No Sync events found in range.")
        return
    for k in events:
        block_num = events[k]['blockNumber']
        self.apply(lp, tkn, user_nm, block_num)

### Listing 9.20: Running the Impermanent Loss Exit Agent

In [19]:
price_threshold = 3000.0
swap_amount = 1.0
pair_address = "0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"
provider_url = "https://mainnet.infura.io/v3/9624e3e5c40f4ac3958b79fa5aa2562d"
platform = Platform.AGNOSTIC
abi_name = JSONContract.UniswapV2Pair

config = PriceThresholdConfig(
    threshold = price_threshold,
    swap_amount = swap_amount,
    pool_address = pair_address,
    provider_url = provider_url,
    platform = platform,
    abi_name = abi_name,
)

agent = PriceThresholdSwapAgent(config)

abi = ABILoad(platform, abi_name)
connect = ConnectW3(provider_url)
connect.apply()

last_block = connect.get_w3().eth.block_number
start_block = last_block - 25

# Grab batch sync events from pool
rEvents = RetrieveEvents(connect, abi)
events = rEvents.apply(EventType.SYNC, address = pair_address, start_block=start_block, end_block=last_block)

agent.apply()
lp_data = agent.get_lp_data()
tkn0 = lp_data.tkn0; tkn1 = lp_data.tkn1; reserves = lp_data.reserves

print("-------------------")
print(f"Agent data")
print("-------------------")
agent.run_batch(tkn0, events)

-------------------
Agent data
-------------------
Block 23729699: Swapped 1.0 USDC for 0.000308236592274451 WETH
Block 23729709: Swapped 1.0 USDC for 0.000308236537888041 WETH
Block 23729710: Swapped 1.0 USDC for 0.000308236483501646 WETH


### Listing 9.21: Running price swap agent

In [20]:
price_threshold = 3000.0
swap_amount = 1.0
pair_address = "0xB4e16d0168e52d35CaCD2c6185b44281Ec28C9Dc"
provider_url = "https://mainnet.infura.io/v3/9624e3e5c40f4ac3958b79fa5aa2562d"
platform = Platform.AGNOSTIC
abi_name = JSONContract.UniswapV2Pair

config = PriceThresholdConfig(
    threshold = price_threshold,
    swap_amount = swap_amount,
    pool_address = pair_address,
    provider_url = provider_url,
    platform = platform,
    abi_name = abi_name,
)

agent = PriceThresholdSwapAgent(config)

abi = ABILoad(platform, abi_name)
connect = ConnectW3(provider_url)
connect.apply()

last_block = connect.get_w3().eth.block_number
start_block = last_block - 100

# Grab batch sync events from pool
rEvents = RetrieveEvents(connect, abi)
events = rEvents.apply(EventType.SYNC, address = pair_address, start_block=start_block, end_block=last_block)

agent.apply()
lp_data = agent.get_lp_data()
tkn0 = lp_data.tkn0; tkn1 = lp_data.tkn1; reserves = lp_data.reserves

print("-------------------")
print(f"Agent data")
print("-------------------")
agent.run_batch(tkn0, events)

-------------------
Agent data
-------------------
Block 23729623: Swapped 1.0 USDC for 0.000307761070148006 WETH
Block 23729630: Swapped 1.0 USDC for 0.000307761015887335 WETH
Block 23729662: Swapped 1.0 USDC for 0.000307760961626679 WETH
Block 23729670: Swapped 1.0 USDC for 0.000307760907366038 WETH
Block 23729672: Swapped 1.0 USDC for 0.00030776085310541 WETH
Block 23729699: Swapped 1.0 USDC for 0.000307760798844797 WETH
Block 23729709: Swapped 1.0 USDC for 0.000307760744584199 WETH
Block 23729710: Swapped 1.0 USDC for 0.000307760690323615 WETH
Block 23729714: Swapped 1.0 USDC for 0.000307760636063045 WETH
Block 23729715: Swapped 1.0 USDC for 0.000307760581802489 WETH


### Listing: 9.22 Listening for Swap events

In [21]:
from web3 import Web3
import json, time

def handle_event(event):
    start_block = event['blockNumber']
    lp = agent.prime_pool_state(start_block, 'user')
    
    reserve0 = event['args']['reserve0']
    reserve1 = event['args']['reserve1']
    block_num = event['blockNumber']
    event_price = agent.calc_price(reserve0, reserve1, tkn1_over_tkn0 = True)
    
    if event_price >= agent.config.threshold:
        agent.execute_action(lp, tkn0, event_price, block_num)

print("Listening for Swap events...")

try:
    for event_num in events:
        handle_event(events[event_num])
        time.sleep(2)  # short polling interval
except Exception as e:
    print("Error:", e)
    time.sleep(5)

Listening for Swap events...
Block 23729623: Swapped 1.0 USDC for 0.000307761070148006 WETH
Block 23729630: Swapped 1.0 USDC for 0.000307795705932498 WETH
Block 23729662: Swapped 1.0 USDC for 0.000307795463950945 WETH
Block 23729670: Swapped 1.0 USDC for 0.000307785424289285 WETH
Block 23729672: Swapped 1.0 USDC for 0.000307782348265456 WETH
Block 23729699: Swapped 1.0 USDC for 0.000308236592274451 WETH
Block 23729709: Swapped 1.0 USDC for 0.000308567306578573 WETH
Block 23729710: Swapped 1.0 USDC for 0.000308967420714762 WETH
Block 23729714: Swapped 1.0 USDC for 0.000309391525648964 WETH
Block 23729715: Swapped 1.0 USDC for 0.000309606348214731 WETH
